In [1]:
from pathlib import Path

# Locate the source PDF in the user's Downloads folder.
downloads_dir = Path.home() / 'Downloads'
pdf_candidates = sorted(downloads_dir.glob('Premdue-*.pdf'))
if not pdf_candidates:
    raise FileNotFoundError(f'No Premdue PDF found in {downloads_dir}')
pdf_path = str(pdf_candidates[0])
print(f'Using PDF: {pdf_path}')

Using PDF: C:\Users\parth\Downloads\Premdue-202608-0111699F.pdf


Read PDF into df

In [2]:
import os
import re
import sys
import pdfplumber
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StructField, StructType, StringType

# pdf_path is selected from the Downloads folder in the previous cell.

# Use the active Windows notebook interpreter for Spark worker processes.
python_executable = sys.executable
os.environ['PYSPARK_PYTHON'] = python_executable
os.environ['PYSPARK_DRIVER_PYTHON'] = python_executable
active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()
spark = (
    SparkSession.builder
    .appName('LICListConverter')
    .config('spark.pyspark.python', python_executable)
    .config('spark.pyspark.driver.python', python_executable)
    .getOrCreate()
)

schema = StructType([
    StructField('Sr no.', IntegerType(), False),
    StructField('Policy No', StringType(), True),
    StructField('Name', StringType(), True),
    StructField('DOC', StringType(), True),
    StructField('Plan/Tm', StringType(), True),
    StructField('Mod', StringType(), True),
    StructField('FUP', StringType(), True),
    StructField('Flg', StringType(), True),
    StructField('InstPrem', DoubleType(), True),
    StructField('Due', IntegerType(), True),
    StructField('GST', DoubleType(), True),
    StructField('Premium', DoubleType(), True),
    StructField('EstCom', DoubleType(), True),
])

records = []

def clean_cell(value):
    return re.sub(r'\s+', ' ', str(value or '')).strip()

def clean_doc(value):
    match = re.search(r'(\d{1,2}/\d{2}/\d{4})', clean_cell(value))
    return match.group(1) if match else clean_cell(value)

def number(value, integer=False):
    value = clean_cell(value).replace(',', '')
    if not value:
        return None
    return int(float(value)) if integer else float(value)

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        tables = page.extract_tables({
            'vertical_strategy': 'lines',
            'horizontal_strategy': 'lines',
            'snap_tolerance': 3,
            'join_tolerance': 3,
            'intersection_tolerance': 3,
        })
        for table in tables:
            for row in table:
                cells = [clean_cell(cell) for cell in row]
                if not cells or not cells[0].isdigit() or len(cells) not in (14, 16):
                    continue
                if len(cells) == 14:
                    serial, policy, name = cells[0:3]
                    doc, plan, mode, fup, flag = cells[4:9]
                    instalment, due, gst = cells[9:12]
                    totals = [cell for cell in cells[12:] if cell]
                else:
                    serial, policy, name = cells[0:3]
                    doc, plan, mode, fup, flag = cells[5:10]
                    instalment, due, gst = cells[10:13]
                    totals = [cell for cell in cells[13:] if cell]
                if len(totals) < 2:
                    continue
                records.append({
                    'Sr no.': number(serial, integer=True),
                    'Policy No': policy,
                    'Name': name,
                    'DOC': clean_doc(doc),
                    'Plan/Tm': plan,
                    'Mod': mode,
                    'FUP': fup,
                    'Flg': flag,
                    'InstPrem': number(instalment),
                    'Due': number(due, integer=True),
                    'GST': number(gst),
                    'Premium': number(totals[0]),
                    'EstCom': number(totals[1]),
                })

pdf_df = spark.createDataFrame(records, schema=schema)
print(f'Read {pdf_df.count()} policy rows')
pdf_df.orderBy(F.col('`Sr no.`')).show(5, truncate=False)

Read 277 policy rows


+------+---------+-----------------------+----------+-------+---+-------+---+--------+---+---+-------+-------+
|Sr no.|Policy No|Name                   |DOC       |Plan/Tm|Mod|FUP    |Flg|InstPrem|Due|GST|Premium|EstCom |
+------+---------+-----------------------+----------+-------+---+-------+---+--------+---+---+-------+-------+
|1     |913952675|NAYNA PANKAJ ZADE      |28/02/2022|936/25 |Yly|02/2026|   |23643.0 |1  |0.0|23643.0|1182.15|
|2     |913954417|RAJESH KANTHIRAM SALAM |28/02/2022|914/21 |Yly|02/2026|   |4882.0  |1  |0.0|4882.0 |244.1  |
|3     |977636620|DURGA ASHOK KHANTE     |28/02/2011|165/20 |Hly|02/2026|   |6065.0  |2  |0.0|12130.0|606.5  |
|4     |979177225|RAGHUNATH ZIBAL UIKE   |28/08/2015|814/21 |Hly|02/2026|   |2562.0  |2  |0.0|5124.0 |256.2  |
|5     |979653836|SACHIN KRUSHNARAO JUMDE|28/02/2017|841/24 |Qly|02/2026|   |1540.0  |3  |0.0|4620.0 |231.0  |
+------+---------+-----------------------+----------+-------+---+-------+---+--------+---+---+-------+-------+
o

Apply transformation

In [3]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Remove life-assured markers and normalize OCR punctuation before name matching.
name_text = F.trim(F.regexp_replace(F.col('Name'), r'(?i)\s*(?:\(\s*l\.?\s*a\.?\s*\)|\[\s*l\.?\s*a\.?\s*\]|\.\.\.\s*l\.?\s*a\.?(?=$|[\s,])|(?<![A-Za-z])l\.?\s*a\.?(?=$|[\s,]))', ''))
name_text = F.trim(F.regexp_replace(name_text, r'(?<=[A-Za-z])\.(?=[A-Za-z])', ' '))
name_text = F.trim(F.regexp_replace(name_text, r'\s+', ' '))

# Match the first and last name tokens, then format them as Surname Givenname.
given_name = F.regexp_extract(name_text, r'^([A-Za-z]+)', 1)
surname = F.regexp_extract(name_text, r'([A-Za-z]+)\s*$', 1)
normalized_name = F.concat_ws(' ', surname, given_name)
pdf_df = pdf_df.withColumn('Name', F.initcap(normalized_name))

# Keep current-month and six-month-back FUP records, with six-month-back rows first.
fup_month_date = F.to_date(F.concat(F.lit('01/'), F.col('FUP')), 'dd/MM/yyyy')
doc_text = F.regexp_replace(F.col('DOC'), r'(?i)i', '1')
doc_day = F.regexp_extract(doc_text, r'^\D*(\d{1,2})/', 1)
pdf_df = pdf_df.withColumn('FUP_month_date', fup_month_date)
today = F.current_date()
six_months_back = F.add_months(today, -6)
current_month = (
    (F.year('FUP_month_date') == F.year(today)) &
    (F.month('FUP_month_date') == F.month(today))
)
six_month_back = (
    (F.year('FUP_month_date') == F.year(six_months_back)) &
    (F.month('FUP_month_date') == F.month(six_months_back))
)
pdf_df = pdf_df.filter(
    (current_month | six_month_back) &
    (~F.trim(F.col('Mod')).eqNullSafe('Mly'))
)
pdf_df = pdf_df.withColumn(
    'FUP_priority',
    F.when(six_month_back, F.lit(0)).otherwise(F.lit(1)),
)
pdf_df = pdf_df.withColumn(
    'FUP',
    F.concat(
        F.lpad(doc_day, 2, '0'),
        F.lit('-'),
        F.date_format('FUP_month_date', 'MMM'),
    ),
).drop('FUP_month_date')

# Use InstPrem as the final Premium value and mark FY, MT, and LP records.
pdf_df = pdf_df.withColumn('Premium', F.regexp_replace(F.col('InstPrem').cast('string'), r'\.0$', ''))
pdf_df = pdf_df.withColumn(
    'Premium',
    F.when(
        F.upper(F.coalesce(F.col('Flg'), F.lit(''))).rlike('FY|MT|LP'),
        F.concat(F.lit('*'), F.col('Premium')),
    ).otherwise(F.col('Premium')),
)

# Make the mode column hold uppercase initials under alias M.
pdf_df = (
    pdf_df
    .withColumn('Mod', F.upper(F.substring(F.trim(F.col('Mod')), 1, 1)))
    .withColumnRenamed('Mod', 'M')
)

keep_columns = ['SN', 'FUP', 'DOC', 'M', 'Policy No', 'Premium', 'Name']

# Restart SN at 1 for each FUP-priority group.
sort_window = Window.partitionBy('FUP_priority').orderBy(
    F.split(F.coalesce(F.col('Name'), F.lit('')), ' ').getItem(0),
)
pdf_df = pdf_df.withColumn('SN', F.row_number().over(sort_window))
pdf_df = pdf_df.select(*[F.col(column) for column in keep_columns] + [F.col('FUP_priority')])
pdf_df.show(n=pdf_df.count(), truncate=False)

+---+------+----------+---+---------+-------+--------------------+------------+
|SN |FUP   |DOC       |M  |Policy No|Premium|Name                |FUP_priority|
+---+------+----------+---+---------+-------+--------------------+------------+
|1  |28-Feb|28/02/2017|Q  |979653836|1540   |Jumde Sachin        |0           |
|2  |28-Feb|28/02/2011|H  |977636620|6065   |Khante Durga        |0           |
|3  |28-Feb|28/02/2022|Y  |913954417|4882   |Salam Rajesh        |0           |
|4  |28-Feb|28/08/2015|H  |979177225|2562   |Uike Raghunath      |0           |
|5  |28-Feb|28/02/2022|Y  |913952675|23643  |Zade Nayna          |0           |
|1  |28-Aug|28/11/2022|Q  |936002705|2300   |Aasvale Laxmi       |1           |
|2  |28-Aug|28/02/2023|Q  |937128640|3005   |Aherwar Jivan       |1           |
|3  |28-Aug|28/11/2006|Q  |975505591|*587   |Aherwar Jeevan      |1           |
|4  |28-Aug|28/08/2009|Y  |976643943|12010  |Anasane Archana     |1           |
|5  |28-Aug|28/08/2007|Y  |975945025|*49

In [4]:
from pathlib import Path
from datetime import datetime
from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_TAB_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

output_path = Path(pdf_path).with_name('LIC_Premium_Due_List.docx')
rows = pdf_df.orderBy('FUP_priority', 'SN').collect()

# Keep the printed layout stable: Name spans surname and given-name cells.
columns = ['SN', 'F.U.P.', 'D.O.C.', 'M', 'Policy No.', 'Premium', 'Name', '']

# Cell padding uses twips; the default left padding keeps table content off the border.
table_left_padding = 80
name_left_padding = 750
premium_right_padding = 400

def set_cell_shading(cell, fill):
    properties = cell._tc.get_or_add_tcPr()
    shading = properties.find(qn('w:shd'))
    if shading is None:
        shading = OxmlElement('w:shd')
        properties.append(shading)
    shading.set(qn('w:fill'), fill)

def set_cell_margins(cell, top=35, start=table_left_padding, bottom=35, end=45):
    properties = cell._tc.get_or_add_tcPr()
    margins = properties.first_child_found_in('w:tcMar')
    if margins is None:
        margins = OxmlElement('w:tcMar')
        properties.append(margins)
    for side, value in [('top', top), ('start', start), ('bottom', bottom), ('end', end)]:
        element = margins.find(qn(f'w:{side}'))
        if element is None:
            element = OxmlElement(f'w:{side}')
            margins.append(element)
        element.set(qn('w:w'), str(value))
        element.set(qn('w:type'), 'dxa')

def remove_shared_vertical_border(left_cell, right_cell):
    for cell, side in [(left_cell, 'right'), (right_cell, 'left')]:
        properties = cell._tc.get_or_add_tcPr()
        borders = properties.first_child_found_in('w:tcBorders')
        if borders is None:
            borders = OxmlElement('w:tcBorders')
            properties.append(borders)
        border = borders.find(qn(f'w:{side}'))
        if border is None:
            border = OxmlElement(f'w:{side}')
            borders.append(border)
        border.set(qn('w:val'), 'nil')

def set_row_bottom_border(row, color='292929', size='18'):
    for cell in row.cells:
        properties = cell._tc.get_or_add_tcPr()
        borders = properties.first_child_found_in('w:tcBorders')
        if borders is None:
            borders = OxmlElement('w:tcBorders')
            properties.append(borders)
        border = borders.find(qn('w:bottom'))
        if border is None:
            border = OxmlElement('w:bottom')
            borders.append(border)
        border.set(qn('w:val'), 'single')
        border.set(qn('w:sz'), size)
        border.set(qn('w:space'), '0')
        border.set(qn('w:color'), color)

def set_cell_text(cell, value, font_size=10, bold=False, color='000000', align=WD_ALIGN_PARAGRAPH.LEFT):
    cell.text = '' if value is None else str(value)
    paragraph = cell.paragraphs[0]
    paragraph.alignment = align
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)
    for run in paragraph.runs:
        run.font.name = 'Arial'
        run.font.size = Pt(font_size)
        run.bold = bold
        run.font.color.rgb = RGBColor.from_string(color)

def split_name(value):
    parts = str(value or '').split(maxsplit=1)
    return parts[0], parts[1] if len(parts) > 1 else ''

document = Document()
section = document.sections[0]
section.orientation = WD_ORIENT.PORTRAIT
section.page_width = Inches(8.27)
section.page_height = Inches(11.69)
section.left_margin = Inches(0.28)
section.right_margin = Inches(0.28)
section.top_margin = Inches(0.3)
section.bottom_margin = Inches(0.3)

month_text = next((row['FUP'] for row in rows if row['FUP']), '')
month_label = f"{datetime.strptime(month_text[-3:], '%b').strftime('%B')} {datetime.now().year}" if month_text else ''
heading = document.add_paragraph()
heading.paragraph_format.space_after = Pt(5)
heading.paragraph_format.tab_stops.add_tab_stop(Inches(3.85), WD_TAB_ALIGNMENT.CENTER)
heading.paragraph_format.tab_stops.add_tab_stop(Inches(7.7), WD_TAB_ALIGNMENT.RIGHT)
run = heading.add_run('\tPremium Due list\t')
run.font.name = 'Arial'
run.font.size = Pt(16)
run.bold = True
run = heading.add_run(month_label)
run.font.name = 'Arial'
run.font.size = Pt(12)
run.bold = True

table = document.add_table(rows=1, cols=8)
table.style = 'Table Grid'
table.alignment = WD_TABLE_ALIGNMENT.RIGHT
table.autofit = False
widths = [0.34, 0.58, 0.82, 0.3, 0.862, 1.181, 1.527, 2.0]
header_cells = table.rows[0].cells
for index, width in enumerate(widths):
    header_cells[index].width = Inches(width)
    header_cells[index].vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
    set_cell_margins(header_cells[index])
    set_cell_shading(header_cells[index], '292929')

for index, column in enumerate(columns):
    if index == 6:
        merged_name_cell = header_cells[6].merge(header_cells[7])
        set_cell_text(merged_name_cell, 'Name', font_size=10, bold=True, color='FFFFFF', align=WD_ALIGN_PARAGRAPH.CENTER)
        break
    set_cell_text(header_cells[index], column, font_size=10, bold=True, color='FFFFFF', align=WD_ALIGN_PARAGRAPH.CENTER)

for position, row in enumerate(rows):
    cells = table.add_row().cells
    surname, given_name = split_name(row['Name'])
    values = [row['SN'], row['FUP'], row['DOC'], row['M'], row['Policy No'], row['Premium'], surname, given_name]
    for index, value in enumerate(values):
        cells[index].width = Inches(widths[index])
        cells[index].vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
        if index == 5:
            set_cell_margins(cells[index], top=25, bottom=25, end=premium_right_padding)
            alignment = WD_ALIGN_PARAGRAPH.RIGHT
        elif index >= 6:
            set_cell_margins(cells[index], top=25, bottom=25, start=name_left_padding)
            alignment = WD_ALIGN_PARAGRAPH.LEFT
        else:
            set_cell_margins(cells[index], top=25, bottom=25)
            alignment = WD_ALIGN_PARAGRAPH.CENTER
        set_cell_text(cells[index], value, font_size=10, align=alignment)
    remove_shared_vertical_border(cells[6], cells[7])
    is_last_six_month_row = (
        row['FUP_priority'] == 0 and
        (position == len(rows) - 1 or rows[position + 1]['FUP_priority'] != 0)
    )
    if is_last_six_month_row:
        set_row_bottom_border(table.rows[-1])

header_properties = table.rows[0]._tr.get_or_add_trPr()
repeat_header = OxmlElement('w:tblHeader')
repeat_header.set(qn('w:val'), 'true')
header_properties.append(repeat_header)

document.save(output_path)
print(f'Created Word file: {output_path}')
print(f'Exported {len(rows)} rows with left-padded table content')

Created Word file: C:\Users\parth\Downloads\LIC_Premium_Due_List.docx
Exported 156 rows with left-padded table content
